In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM, GPTNeoConfig, GPTNeoForCausalLM

cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = 1
seed = 3407
batch_size = 64
window_size = 256
lr = 1e-3

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
    

tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
model = AutoModelForCausalLM.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")

# Initializing a model (with random weights) from the EleutherAI/gpt-neo-1.3B style configuration
model = GPTNeoForCausalLM(model.config)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in model: {num_params}")

KeyboardInterrupt: 

In [ ]:
# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")
else:
    print("Warning: HF_TOKEN not found in .env file")

# Set your HuggingFace username/organization
HF_USERNAME = os.getenv('HF_USERNAME', 'your-username')  # Change this to your HF username
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

# Global variable to track data indices used since last checkpoint
data_tracker = {
    'train_indices': [],
    'train_data': [],
    'val_indices': [],
    'val_data': []
}

def reset_data_tracker():
    """Reset the data tracker for a new checkpoint period"""
    global data_tracker
    data_tracker = {
        'train_indices': [],
        'train_data': [],
        'val_indices': [],
        'val_data': []
    }

def save_checkpoint(model, optimizer, updates, checkpoint_name, data_tracker_state=None):
    """
    Save checkpoint to HuggingFace Hub in subfolder structure
    Args:
        model: The model to save
        optimizer: The optimizer state to save
        updates: Number of training updates/steps
        checkpoint_name: Name for this checkpoint (e.g., "checkpoint-1000")
        data_tracker_state: Dictionary containing train/val indices and data used since last checkpoint
    """
    # Get the base model if wrapped in DataParallel
    model_to_save = model.module if hasattr(model, 'module') else model
    
    # Create checkpoint subfolder structure: temp_dir/checkpoint-{step}/
    checkpoint_folder = f"checkpoint-{updates}"
    temp_dir = f"temp_{checkpoint_folder}"
    checkpoint_path = os.path.join(temp_dir, checkpoint_folder)
    os.makedirs(checkpoint_path, exist_ok=True)
    
    # Save model to checkpoint subfolder (uses safetensors by default)
    model_to_save.save_pretrained(checkpoint_path)
    
    # Save optimizer state (no model weights, just optimizer)
    optimizer_dict = {
        'optimizer_state_dict': optimizer.state_dict(),
        'updates': updates,
    }
    torch.save(optimizer_dict, os.path.join(checkpoint_path, 'optimizer.pt'))
    
    # Save data tracker if provided
    if data_tracker_state is not None:
        with open(os.path.join(checkpoint_path, 'data_tracker.json'), 'w') as f:
            json.dump(data_tracker_state, f, indent=2)
    
    # Push to HuggingFace Hub
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    commit_message = f"Checkpoint at step {updates}"
    
    try:
        api = HfApi()
        
        # Create repository if it doesn't exist
        try:
            from huggingface_hub import create_repo
            create_repo(
                repo_id=repo_name,
                private=True,  # Set to False if you want public repo
                exist_ok=True  # Don't error if repo already exists
            )
            print(f"Repository {repo_name} ready")
        except Exception as e:
            print(f"Note: {e}")
        
        # Upload entire checkpoint folder
        api.upload_folder(
            folder_path=checkpoint_path,
            path_in_repo=checkpoint_folder,
            repo_id=repo_name,
            commit_message=commit_message,
        )
        
        print(f"Checkpoint saved to HuggingFace: {repo_name}/{checkpoint_folder}")
        logging.info(f"Checkpoint saved to HuggingFace: {repo_name}/{checkpoint_folder} at step {updates}")
        if data_tracker_state is not None:
            print(f"Saved {len(data_tracker_state['train_data'])} training samples, {len(data_tracker_state['val_data'])} validation samples")
            logging.info(f"Saved {len(data_tracker_state['train_data'])} training samples, {len(data_tracker_state['val_data'])} validation samples")
    except Exception as e:
        print(f"Error saving checkpoint to HuggingFace: {e}")
        logging.error(f"Error saving checkpoint to HuggingFace: {e}")
        import traceback
        traceback.print_exc()
    finally:
        # Clean up temporary directory
        import shutil
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

def load_checkpoint(model, optimizer, checkpoint_step):
    """
    Load checkpoint from HuggingFace Hub
    Args:
        model: The model to load weights into
        optimizer: The optimizer to load state into
        checkpoint_step: Checkpoint step number to load (e.g., 1000, 2000)
    Returns:
        updates: Number of training updates/steps from checkpoint
    """
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    checkpoint_folder = f"checkpoint-{checkpoint_step}"
    
    try:
        from huggingface_hub import hf_hub_download, repo_exists
        
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist")
            logging.error(f"Repository {repo_name} does not exist")
            return 0
        
        # Get the base model if wrapped in DataParallel
        model_to_load = model.module if hasattr(model, 'module') else model
        
        # Load model from checkpoint subfolder
        print(f"Loading model from {repo_name}/{checkpoint_folder}...")
        loaded_model = GPTNeoForCausalLM.from_pretrained(
            repo_name,
            subfolder=checkpoint_folder
        )
        model_to_load.load_state_dict(loaded_model.state_dict())
        
        # Download and load optimizer state
        optimizer_path = hf_hub_download(
            repo_id=repo_name,
            filename=f"{checkpoint_folder}/optimizer.pt"
        )
        optimizer_dict = torch.load(optimizer_path)
        
        optimizer.load_state_dict(optimizer_dict['optimizer_state_dict'])
        updates = optimizer_dict['updates']
        
        print(f"Checkpoint loaded from HuggingFace: {repo_name}/{checkpoint_folder} (step {updates})")
        logging.info(f"Checkpoint loaded from HuggingFace: {repo_name}/{checkpoint_folder} at step {updates}")
        
        return updates
    except FileNotFoundError as e:
        print(f"Error: Checkpoint {checkpoint_folder} not found in {repo_name}")
        print(f"Details: {e}")
        logging.error(f"Checkpoint {checkpoint_folder} not found in {repo_name}: {e}")
        return 0
    except Exception as e:
        print(f"Error loading checkpoint from HuggingFace: {e}")
        logging.error(f"Error loading checkpoint from HuggingFace: {e}")
        import traceback
        traceback.print_exc()
        return 0

class TrackingDataset(Dataset):
    """
    Wrapper dataset that tracks which samples are accessed
    """
    def __init__(self, dataset, mode='train'):
        self.dataset = dataset
        self.mode = mode
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # Track this access
        global data_tracker
        if self.mode == 'train':
            data_tracker['train_indices'].append(idx)
            data_tracker['train_data'].append({
                'index': idx,
                'text': self.dataset[idx]['text']
            })
        else:
            data_tracker['val_indices'].append(idx)
            data_tracker['val_data'].append({
                'index': idx,
                'text': self.dataset[idx]['text']
            })
        
        return self.dataset[idx]

# ============================================
# CONVERGENCE MONITORING FUNCTIONS
# ============================================

@torch.no_grad()
def compute_grad_metrics(model):
    """
    Compute lightweight gradient metrics for convergence monitoring.
    Returns:
        dict with global_norm, max_abs, and param_norm
    """
    total_grad_sq = 0.0
    max_abs_grad = 0.0
    total_param_sq = 0.0
    
    # Handle DataParallel wrapper
    model_to_check = model.module if hasattr(model, 'module') else model
    
    for p in model_to_check.parameters():
        if p.grad is None:
            continue
        
        g = p.grad.detach()
        
        # Global gradient stats
        total_grad_sq += g.pow(2).sum().item()
        max_abs_grad = max(max_abs_grad, g.abs().max().item())
        
        # Parameter norm (for relative metrics)
        total_param_sq += p.detach().pow(2).sum().item()
    
    global_norm = (total_grad_sq ** 0.5)
    param_norm = (total_param_sq ** 0.5)
    
    return {
        'grad_global_norm': global_norm,
        'grad_max_abs': max_abs_grad,
        'grad_to_param_ratio': global_norm / (param_norm + 1e-12)
    }

@torch.no_grad()
def compute_update_metrics(model, prev_params):
    """
    Compute parameter update metrics (call after optimizer.step()).
    Args:
        model: Current model with updated parameters
        prev_params: List of parameter tensors before optimizer.step()
    Returns:
        dict with update_norm and relative_update
    """
    # Handle DataParallel wrapper
    model_to_check = model.module if hasattr(model, 'module') else model
    
    deltas_sq = 0.0
    params_sq = 0.0
    
    for p_prev, p in zip(prev_params, model_to_check.parameters()):
        delta = (p - p_prev)
        deltas_sq += delta.pow(2).sum().item()
        params_sq += p.pow(2).sum().item()
    
    update_norm = (deltas_sq ** 0.5)
    param_norm = (params_sq ** 0.5)
    
    return {
        'update_norm': update_norm,
        'update_relative': update_norm / (param_norm + 1e-12)
    }

print("Checkpoint and convergence monitoring functions loaded")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace
Checkpoint and convergence monitoring functions loaded


In [ ]:
import random
# Set up logger
current_time = datetime.now().strftime("%m%d_%H%M%S")
import os
if not os.path.exists("logs"):
    os.makedirs("logs")

log_filename = f"logs/training_{cfg_param}_{current_time}.log"
logging.basicConfig(filename=log_filename, level=logging.INFO,
                    format='%(asctime)s %(levelname)s: %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

# Load dataset and tokenizer
model_name = 'roneneldan/TinyStories'
dataset = load_dataset(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Wrap datasets with tracking
train_dataset = TrackingDataset(dataset['train'], mode='train')
valid_dataset = TrackingDataset(dataset['validation'], mode='val')

# Create deterministic generators for shuffling
train_generator = torch.Generator()
train_generator.manual_seed(seed)
valid_generator = torch.Generator()
valid_generator.manual_seed(seed)

# Instantiate dataloader with deterministic shuffling
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    generator=train_generator,
    worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id)
)
for batch in train_loader:
    print(batch['text'][0])
    break

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    generator=valid_generator,
    worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id)
)

# Instantiate model and optimizer

def estimate_loss(model, tokenizer, valid_loader, device='cuda'):
    model.eval()
    with torch.no_grad():
        losses = torch.zeros(40)
        for k,batch in enumerate(valid_loader):
            tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length = 256, truncation = True)['input_ids'].to(device)
            outputs = model(tokenized, labels=tokenized)
            loss = outputs.loss
            if torch.cuda.device_count() > 1:
                loss = loss.mean()
            losses[k] = loss.item()
            if k == 40 - 1 :
                break
    model.train()
    return losses.mean()


if torch.cuda.device_count() > 1:
    # if multiple gpus on single machine
    model = nn.DataParallel(model)
model.to(device)
optim = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.95))

# ============================================
# RESUME TRAINING CONFIGURATION
# ============================================
resume_training = True  # Set to True to continue from a checkpoint
checkpoint_to_load = 66242  # The checkpoint step to resume from
wandb_run_id = "kqz0ks8b"  # The WandB run ID to continue logging to
start_epoch = 2  # Which epoch number we're starting from (0-indexed, so 1 = second epoch)

updates = 0
hf_repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
if resume_training:
    logging.info(f"Resuming training from checkpoint {checkpoint_to_load}")
    updates = load_checkpoint(model, optim, checkpoint_to_load)
    print(f"Resumed from checkpoint {checkpoint_to_load}, continuing from step {updates}")

# Setup weights & biases
if resume_training:
    # Resume existing run
    run = wandb.init(
        project="gpt-tinystories",
        id=wandb_run_id,
        resume="allow",
        config={
            "cfg_param": cfg_param,
            "learning_rate": lr,
            "batch_size": batch_size,
            "hf_repo": hf_repo_name,
            "log_filename": log_filename,
            "seed": seed,
            "epochs": epochs,
            "resumed_from_step": checkpoint_to_load
        },
    )
    print(f"Resumed WandB run: {wandb_run_id}")
else:
    # Start new run
    run = wandb.init(
        project="gpt-tinystories",
        name=f"gpt-tinystories-{cfg_param}-{current_time}",
        config={
            "cfg_param": cfg_param,
            "learning_rate": lr,
            "batch_size": batch_size,
            "hf_repo": hf_repo_name,
            "log_filename": log_filename,
            "seed": seed,
            "epochs": epochs
        },
    )
    
logging.info(f"cfg_param: {cfg_param}, lr: {lr}, batch_size: {batch_size}, hf_repo: {hf_repo_name}, log_filename: {log_filename}, seed: {seed}, epochs: {epochs}")

# Reset data tracker at start
reset_data_tracker()

# Training loop
for epoch in range(start_epoch, start_epoch + epochs):
    logging.info(f"Epoch: {epoch+1}")
    for batch in tqdm(train_loader):
        # Store parameters before update (for computing update metrics)
        prev_params = [p.detach().clone() for p in (model.module if hasattr(model, 'module') else model).parameters()]
        
        optim.zero_grad()
        tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length=256, truncation=True)['input_ids'].to(device)
        outputs = model(tokenized, labels=tokenized)
        loss = outputs.loss
        if torch.cuda.device_count() > 1:
            loss = loss.mean()
        loss.backward()
        
        # Compute gradient metrics (before optimizer.step())
        grad_metrics = compute_grad_metrics(model)
        
        optim.step()
        updates += 1
        
        # Compute update metrics (after optimizer.step())
        update_metrics = compute_update_metrics(model, prev_params)
        
        if updates % 200 == 0:
            validation_loss = estimate_loss(model, tokenizer, valid_loader)
            tqdm.write(f"Train_{epoch+1}_{updates}: {validation_loss}")
            logging.info(f"Train_{epoch+1}_{updates}: {validation_loss}")
            
            # Log all metrics to WandB
            wandb.log({
                "train_loss": loss.item(),
                "val_loss": validation_loss,
                "grad_global_norm": grad_metrics['grad_global_norm'],
                "grad_max_abs": grad_metrics['grad_max_abs'],
                "grad_to_param_ratio": grad_metrics['grad_to_param_ratio'],
                "update_norm": update_metrics['update_norm'],
                "update_relative": update_metrics['update_relative']
            }, step=updates)
        
        if updates % 1000 == 0:
            # Save checkpoint with data tracker
            save_checkpoint(model, optim, updates, f"checkpoint-{updates}", data_tracker_state=data_tracker)
            # Reset tracker after saving
            reset_data_tracker()
            
    logging.info("TRAINING COMPLETE")
    logging.info("Computing final validation loss..")
    # Validation loop
    with torch.no_grad():
        loss_valid = 0
        for batch in tqdm(valid_loader):
            tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length=512,truncation=True)['input_ids'].to(device)
            outputs = model(tokenized, labels=tokenized)
            loss = outputs.loss
            loss_valid += loss.mean().item()
        logging.info(f"Final validation loss: {loss_valid / len(valid_loader)}")
        save_checkpoint(model, optim, updates, f"checkpoint-final-{updates}", data_tracker_state=data_tracker)
        
        # Log HuggingFace repo to wandb
        wandb.log({"hf_repo": hf_repo_name})
        print(f"Final model saved to HuggingFace: {hf_repo_name}")


Once upon a time, there was a sweet little dog named Spot. Spot loved to play with his ball. One day, he saw a big tree in the park. Spot had an urge to play near the tree. He did not know why, but he felt like something fun would happen there.

Spot ran to the tree and started to play with his ball. He saw a pretty bird up in the tree. He wanted to be friends with the bird. Spot tried to point at the bird with his paw, but the bird did not see him. Spot felt sad, but he did not give up. He knew there was a way to make the bird see him.

The next day, Spot went back to the tree. He had a plan. He found a long stick and held it in his mouth. Spot used the stick to point at the bird. This time, the bird saw him! The bird flew down and played with Spot and his ball. They became the best of friends, and Spot was happy. His urge to play near the tree had led him to a new friend.
Loading model from jrosseruk/gpt-tinystories-8M/checkpoint-33121...


checkpoint-33121/optimizer.pt:   0%|          | 0.00/158M [00:00<?, ?B/s]

Checkpoint loaded from HuggingFace: jrosseruk/gpt-tinystories-8M/checkpoint-33121 (step 33121)
Resumed from checkpoint 33121, continuing from step 33121


wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Resumed WandB run: kqz0ks8b


...

100%|██████████| 344/344 [00:30<00:00, 11.26it/s]


Repository jrosseruk/gpt-tinystories-8M ready


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Checkpoint saved to HuggingFace: jrosseruk/gpt-tinystories-8M/checkpoint-66242
Saved 15463 training samples, 24550 validation samples
Final model saved to HuggingFace: jrosseruk/gpt-tinystories-8M


In [ ]:
# Helper function to load and inspect saved training data from a checkpoint
def load_checkpoint_data(checkpoint_step):
    """
    Load the training/validation data used for a specific checkpoint
    Args:
        checkpoint_step: The checkpoint step number (e.g., 1000, 2000)
    Returns:
        Dictionary with train_data and val_data
    """
    from huggingface_hub import hf_hub_download, repo_exists, list_repo_files
    import json
    
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    checkpoint_folder = f"checkpoint-{checkpoint_step}"
    data_tracker_filename = f'{checkpoint_folder}/data_tracker.json'
    
    try:
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            return None
        
        # Check if checkpoint exists
        files = list_repo_files(repo_id=repo_name)
        checkpoint_files = [f for f in files if f.startswith(checkpoint_folder + '/')]
        
        if not checkpoint_files:
            print(f"Error: Checkpoint {checkpoint_folder} not found in {repo_name}")
            available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
            if available_checkpoints:
                print(f"Available checkpoints: {', '.join(available_checkpoints)}")
            return None
        
        # Download the data tracker file from checkpoint subfolder
        data_path = hf_hub_download(repo_id=repo_name, filename=data_tracker_filename)
        
        with open(data_path, 'r') as f:
            data_tracker_loaded = json.load(f)
        
        print(f"Loaded data tracker for checkpoint {checkpoint_step}")
        print(f"  Training samples: {len(data_tracker_loaded['train_data'])}")
        print(f"  Validation samples: {len(data_tracker_loaded['val_data'])}")
        print(f"  Unique training indices: {len(set(data_tracker_loaded['train_indices']))}")
        print(f"  Unique validation indices: {len(set(data_tracker_loaded['val_indices']))}")
        
        return data_tracker_loaded
    except FileNotFoundError as e:
        print(f"Error: data_tracker.json not found in checkpoint {checkpoint_folder}")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"Error loading checkpoint data: {e}")
        import traceback
        traceback.print_exc()
        return None

# Example usage (uncomment to use):
# data = load_checkpoint_data(1000)
# if data:
#     # Access first training sample
#     print("First training sample:", data['train_data'][0])
#     
#     # Get all training texts
#     train_texts = [sample['text'] for sample in data['train_data']]
#     
#     # Verify reproducibility - check if indices match expected order
#     print("Training indices:", data['train_indices'][:10])

In [ ]:
# ============================================
# TEXT GENERATION / INFERENCE
# ============================================

def load_model_for_inference(repo_name=None, checkpoint_step=None, device='cuda'):
    """
    Load a trained model from HuggingFace for text generation
    
    Args:
        repo_name: Full HuggingFace repo name (e.g., "jrosseruk/gpt-tinystories-8M")
                   If None, uses the current cfg_param to construct repo name
        checkpoint_step: Specific checkpoint step to load (not yet implemented - loads latest)
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The loaded model
        tokenizer: The tokenizer
    """
    if repo_name is None:
        repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    
    print(f"Loading model from {repo_name}...")
    
    try:
        # Load model and tokenizer
        inference_model = GPTNeoForCausalLM.from_pretrained(repo_name)
        inference_tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
        inference_tokenizer.pad_token = inference_tokenizer.eos_token
        
        # Move to device and set to eval mode
        inference_model = inference_model.to(device)
        inference_model.eval()
        
        print(f"Model loaded successfully!")
        return inference_model, inference_tokenizer
    
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None


def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.8, top_k=50, top_p=0.95, num_return_sequences=1, device='cuda'):
    """
    Generate text completion from a prompt
    
    Args:
        model: The trained model
        tokenizer: The tokenizer
        prompt: Text prompt to complete
        max_length: Maximum length of generated text (in tokens)
        temperature: Sampling temperature (higher = more random, lower = more deterministic)
        top_k: Keep only top k tokens with highest probability (0 = disabled)
        top_p: Nucleus sampling - keep top tokens with cumulative probability >= top_p
        num_return_sequences: Number of different completions to generate
        device: Device model is on
    
    Returns:
        List of generated text completions
    """
    model.eval()
    
    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        # Generate
        output_sequences = model.generate(
            input_ids=input_ids,
            max_length=max_length + len(input_ids[0]),
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the generated sequences
    generated_texts = []
    for sequence in output_sequences:
        text = tokenizer.decode(sequence, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts


# ============================================
# EXAMPLE USAGE
# ============================================

# Uncomment to load a model and generate text:

# Load your trained model
inference_model, inference_tokenizer = load_model_for_inference()
# inference_model = model
# inference_tokenizer = tokenizer

if inference_model is not None:
    # Define a prompt
    prompt = "Once upon a time, there was a dinosaur"
    
    print(f"Prompt: {prompt}")
    print("=" * 50)
    
    # Generate completions
    completions = generate_text(
        inference_model, 
        inference_tokenizer, 
        prompt, 
        max_length=150,
        temperature=0.8,
        num_return_sequences=3  # Generate 3 different completions
    )
    
    # Print results
    for i, completion in enumerate(completions, 1):
        print(f"\nCompletion {i}:")
        print(completion)
        print("=" * 50)


print("Text generation functions loaded. Uncomment the example usage block to test!")

Loading model from jrosseruk/gpt-tinystories-8M...
Error loading model: jrosseruk/gpt-tinystories-8M does not appear to have a file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt or flax_model.msgpack.
Text generation functions loaded. Uncomment the example usage block to test!
